In [281]:
from src.data_utils.corpus import Corpus
from src.data_utils.download_retrieval_datasets import *
from src.data_utils.dataset import DocumentDataset
import numpy as np
import pandas as pd

In [278]:
fiqa = Corpus(dataset_name= 'fiqa', base_dir= 'data')

In [286]:
def tfidf(dataset, query): # Computes tf-idf score for a given query in a set of documents

    tfidf_score = {}

    import re
    query = re.sub(r"[^a-z ']+", " ", query.lower()) # clean up query. Makes everything lower case and removes anything that is not a lowercase letter, a space, or an apostrophe with a space.
    query = set(query.split()) # Split into unique words

    # Calculate term frequence (TF) matrix for all document. Each document is a row with the frequencies of each term from the query in said document.
    TF_matrix = np.zeros((len(dataset), len(query)))
    for term in query:
        for i, document in enumerate(dataset):
            TF_matrix[i][list(query).index(term)] = document['text'][0].split(' ').count(term) # Tells us how many times each word appears in each document.
    
    # Calculate document frequency (DF) matrix. Use the fact that True=1, False=0 and sum over the rows.
    DF_matrix = (TF_matrix != 0).sum(axis=0) # tells us how many documents each word appears in.

    for j in range(len(dataset)):
        text = (dataset[j]['text'][0]) # load text
        text = re.sub(r"[^a-z ']+", " ", text.lower()) # clean up text, like with the query
        TFIDF = 0
        for term in query:
            if term not in query:
                continue
            # Calculate TF and DF for a given term.
            if len(text) == 0:
                TF = 0
            else:
                TF = TF_matrix[j:j+1, list(query).index(term)][0]/len(text)
            DF = DF_matrix[list(query).index(term)]

            # Do the inverse (I) calculation on the DF matrix. log(N/n)
            IDF = np.log(len(dataset) / DF)
                
            TFIDF += TF*IDF
        tfidf_score[f'Document {j+1}'] = float(TFIDF)
    
    return sorted(tfidf_score.items(), key=lambda item: item[1], reverse=True)


In [287]:
tfidf(fiqa.val, 'Claim your business, or your business claims you')

[('Document 1', 0.022698738480304088),
 ('Document 90', 0.021053765406191156),
 ('Document 290', 0.01987582797332751),
 ('Document 9', 0.019800486065121376),
 ('Document 38', 0.019637128162772504),
 ('Document 7', 0.01380598383195907),
 ('Document 4', 0.012801327815766539),
 ('Document 5', 0.012767576988880008),
 ('Document 419', 0.012320905267189224),
 ('Document 138', 0.01158740651531989),
 ('Document 460', 0.011488891874614335),
 ('Document 118', 0.011436788234265811),
 ('Document 80', 0.011180326296745157),
 ('Document 93', 0.01109881838641888),
 ('Document 109', 0.010651232243415677),
 ('Document 2', 0.010492043359041317),
 ('Document 189', 0.010278385825320696),
 ('Document 10', 0.010013510057273033),
 ('Document 223', 0.010007017542780093),
 ('Document 134', 0.009509167644393374),
 ('Document 107', 0.009263546801457748),
 ('Document 120', 0.008903600135528997),
 ('Document 367', 0.008850474015342485),
 ('Document 433', 0.008355283384534548),
 ('Document 133', 0.00821222134368343